In [26]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [2]:
# (No edit) Imports
import os
import json

import sys
from pathlib import Path
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
import numpy as np

from art.estimators.classification import PyTorchClassifier
from art.attacks.evasion import ProjectedGradientDescent
from art.attacks.evasion import FastGradientMethod
from art.attacks.evasion import SaliencyMapMethod
from art.attacks.evasion import CarliniL2Method
# from art.attacks.evasion import Carlin
from sklearn.preprocessing import MinMaxScaler

sys.path.append(str(Path.cwd().parents[0]))

import os

import torch
from utils.models import OBU
from utils.functions import get_windowed_data, load_model_checkpoint

from sklearn.metrics import precision_score, recall_score, f1_score

/home/sliu/Documents/GitHub/reu26/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [3]:
# Inputs
# checkpoint_file="../../../saved_models/RandomPos-cenFL-updated.ckpt"
data_file = "../data/RandomPos_0709.csv"
train_perc = 80

norm_trained = False
collapse_output = True # True if using JSMA (maybe CW????)


In [4]:
# (No edit) Load data for og model (very time consuming part)
(x_train, y_train), (x_test, y_test), fed_dataset = get_windowed_data(data_file, 
                                                                      normalize=norm_trained, 
                                                                      train_perc=train_perc)



In [5]:
len(x_test), len(y_test)

(124774, 124774)

In [10]:
import argparse
import json
import sys
import os
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from typing import Dict, Any

import pandas as pd

from MBD_systems.data_structures import Parameters
import MBD_systems.data_processing

In [ ]:
python3 -c "
import time
t0 = time.time()
from MBD_systems.windowed_eval import evaluate_windowed_ids
metrics = evaluate_windowed_ids('data/RandomPos_0709.csv', attacker_code=3, workers=20)
print(metrics)
print(f'elapsed: {time.time()-t0:.1f}s')
"


In [17]:
import time
t0 = time.time()
from MBD_systems.windowed_eval import evaluate_windowed_ids
metrics = evaluate_windowed_ids('../data/RandomPos_0709.csv', attacker_code=3, workers=20)

print(f'elapsed: {time.time()-t0:.1f}s')

Splitting ../data/RandomPos_0709.csv by SenderID into ../data/_sender_split_RandomPos_0709 ...
Found 4067 senders, evaluating windows with 20 workers...
Processed 500/4067 senders
Processed 1000/4067 senders
Processed 1500/4067 senders
Processed 2000/4067 senders
Processed 2500/4067 senders
Processed 3000/4067 senders
Processed 3500/4067 senders
Processed 4000/4067 senders
Processed 4067/4067 senders


elapsed: 22.9s


In [16]:
print(json.dumps(metrics, indent=1))

{
 "tp": 184237,
 "tn": 9254,
 "fp": 430094,
 "fn": 281,
 "accuracy": 0.31014833313564133,
 "precision": 0.2998985888714716,
 "recall": 0.9984771133439556,
 "check_counts": {
  "range_plausibility": 1488846,
  "speed_plausibility": 1789074,
  "sudden_appearance": 274102,
  "position_consistency": 5181632,
  "speed_consistency": 688928,
  "position_speed_consistency": 5244379
 },
 "f1": 0.46125613226028955
}


In [19]:
from MBD_systems.windowed_eval import evaluate_windowed_ids_by_pair
metrics = evaluate_windowed_ids_by_pair('../data/RandomPos_0709.csv', attacker_code=3, workers=20)

Splitting ../data/RandomPos_0709.csv by (SenderID, RecieverID) into ../data/_pair_split_RandomPos_0709 ...
Found 147435 sender/receiver pairs, evaluating windows with 20 workers...
Processed 5000/147435 pairs
Processed 10000/147435 pairs
Processed 15000/147435 pairs
Processed 20000/147435 pairs
Processed 25000/147435 pairs
Processed 30000/147435 pairs
Processed 35000/147435 pairs
Processed 40000/147435 pairs
Processed 45000/147435 pairs
Processed 50000/147435 pairs
Processed 55000/147435 pairs
Processed 60000/147435 pairs
Processed 65000/147435 pairs
Processed 70000/147435 pairs
Processed 75000/147435 pairs
Processed 80000/147435 pairs
Processed 85000/147435 pairs
Processed 90000/147435 pairs
Processed 95000/147435 pairs
Processed 100000/147435 pairs
Processed 105000/147435 pairs
Processed 110000/147435 pairs
Processed 115000/147435 pairs
Processed 120000/147435 pairs
Processed 125000/147435 pairs
Processed 130000/147435 pairs
Processed 135000/147435 pairs
Processed 140000/147435 pairs

In [20]:
metrics

{'tp': 126800,
 'tn': 18871,
 'fp': 283334,
 'fn': 290,
 'accuracy': 0.3393261044270257,
 'precision': 0.3091672477775556,
 'recall': 0.9977181524903611,
 'check_counts': {'range_plausibility': 1026292,
  'speed_plausibility': 1245327,
  'sudden_appearance': 186933,
  'position_consistency': 1025595,
  'speed_consistency': 1133039,
  'position_speed_consistency': 2857179},
 'f1': 0.47205634893452264}

In [ ]:
!python "../MBD_systems/main.py" --input_folder="../data/TRAIN/" --type=0

Starting processing with 24 workers...
0
